**Human Movement Intelligence | Gait Analysis | Presented by: Reza Saadatyar (2026)**

**02 | Alignment and Event Reference**

![Alignment and Event Reference](../../images/02_alignment_and_event_reference.svg)

**Objective:** Construct independently force-derived contact events from same-trial original-time sensor streams and establish participant roles.

**Dataset:** Camargo et al. (2021), treadmill/level-ground walking where original-time IMU and independent force reference are usable.

**Notebook scope:** Implemented analysis code; original-data results will be added only after execution and source eligibility confirmation.


**00. Colab Repository Setup**


In [ ]:
# ======================== 00. Colab Repository Setup =========================

# -------------------------- 1. Repository Checkout ---------------------------
from pathlib import Path
import subprocess

repo_url = "https://github.com/RezaSaadatyar/human-movement-intelligence.git"
repo_path = Path("/content") / repo_url.rsplit("/", 1)[-1].removesuffix(".git")
if not repo_path.exists():
    subprocess.run(["git", "clone", "-q", repo_url, str(repo_path)], check=True)

# ---------------------------- 2. Open Repository -----------------------------
%cd {repo_path}


**01. Imports and Research Environment**

Use the established shared utilities and the Camargo-specific alignment/reference functions. No source download or model training is triggered by importing the study.


In [ ]:
# =================== 01. Imports and Research Environment ====================

# ------------------------ 1. Study Environment Setup -------------------------
from python_modules.core.colab_setup import setup_colab
root = setup_colab("gait_analysis")
NOTEBOOK_NUMBER = 2

# ----------------------- 2. MATLAB Table Reader Setup ------------------------
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("matio") is None:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "mat-io==1.0.2"],
        check=True,
    )


# ----------------- 3. Scientific Computing and Data Analysis -----------------
from pathlib import Path
import os
import re
import numpy as np
import pandas as pd

# ------------------- 4. Visualisation and Notebook Display -------------------
import matplotlib.pyplot as plt
from IPython.display import display
from core.plot_style import plot_style, save_figure
from core.output_download import download_outputs_from_colab

plot_style(
    font_family="DejaVu Sans", title_size=12.5, title_weight="normal",
    label_size=11.5, label_weight="normal", tick_size=11,
    legend_size=10, figure_dpi=100, figure_size=(9, 4),
)

# ------------------------ 5. Study Research Utilities ------------------------
from camargo_source import read_sensor_table, validate_timebase
from gait_alignment import mapped_series, validated_overlap
from gait_reference import force_contact_events
from gait_experiments import save_prepared_trial
from gait_evaluation import participant_roles, development_folds

# ---------------------- 6. Source and Output Locations -----------------------
study_dir = root / "gait_analysis"
source_root = Path(os.environ.get("CAMARGO_DATA_ROOT",
                                  root / "data/raw/camargo"))
output_dir = study_dir / "output"
cache_dir = root / "data/processed/camargo/paired_trials"
output_dir.mkdir(parents=True, exist_ok=True)


**02. Review and Explicitly Set Sensor/Reference Contract**

Populate the following from Notebook 01's **original observed column list**. The exact gyroscope axis, force-plate channel, physical units, corresponding foot, and time-origin relationship must be verified. The intentionally unset options cannot be guessed from Camargo's dataset title.


In [ ]:
# ======================== 02. Review Sensor Contract =========================

# --------------------- 1. Stage Inputs and Main Analysis ---------------------
# User-reviewed source-specific contract: set only after inspecting Notebook 01.
GYRO_COLUMN = None             # e.g. the VERIFIED foot or shank gyroscope axis
VERTICAL_FORCE_COLUMN = None   # verified RIGHT/LEFT vertical treadmill force
FOOT_SIDE = None               # "right" or "left", matched to both sensors
FORCE_UPWARD_SIGN = None       # +1 or -1 in original Newton sign convention
TIME_UNIT_IN_SECONDS = None    # conversion factor confirmed from source
IMU_FORCE_TIMEBASE_CONFIRMED = False
REFERENCE_THRESHOLD_N = 20.0  # preregistered candidate; document any change
ANALYSED_CONDITIONS = {"treadmill"}  # separate experimental context from overground
if (not GYRO_COLUMN or not VERTICAL_FORCE_COLUMN
    or FOOT_SIDE not in {"right", "left"}
    or FORCE_UPWARD_SIGN not in (-1, 1)
    or TIME_UNIT_IN_SECONDS is None
    or not IMU_FORCE_TIMEBASE_CONFIRMED):
    raise ValueError(
        "Notebook 02 requires reviewed gyro axis, vertical-force column, "
        "foot side, force sign, physical timestamp unit and independent "
        "timebase confirmation. Use Notebook 01's source-column inspection "
        "and dataset acquisition documentation; do not infer these values."
    )
pairs = pd.read_csv(output_dir / "01_candidate_pairs.csv")
pairs = pairs.loc[pairs.candidate_pair & pairs.condition.isin(ANALYSED_CONDITIONS)]
print("Candidate within-condition pairs:", len(pairs))
if pairs.empty:
    raise ValueError("No candidate treadmill pairs; inspect Notebook 01 inventory.")


**03. Construct Force Contact References on Native Trial Time**

For each pair: validate native time axes, choose the reviewed gyroscope and force columns, extract physically interpreted contact events, exclude edge-censored/invalid trials, and save aligned numeric inputs for downstream notebooks. Force is never used to produce IMU predictions.


In [ ]:
# ====================== 03. Construct Force References =======================

# --------------------- 1. Stage Inputs and Main Analysis ---------------------
recorded, exclusions, event_rows = [], [], []
used_keys = set()
for row in pairs.itertuples(index=False):
    key = "_".join(re.sub(r"[^A-Za-z0-9]+", "_", str(v)).strip("_")
                   for v in (row.subject_id, row.session_id,
                             row.condition, row.trial_id))
    if key in used_keys:
        raise ValueError(f"Duplicate trial identity: {key}")
    used_keys.add(key)
    try:
        imu = read_sensor_table(source_root / row.imu_path)
        fp = read_sensor_table(source_root / row.fp_path)
        ti = validate_timebase(imu, "IMU") * float(TIME_UNIT_IN_SECONDS)
        tf = validate_timebase(fp, "force") * float(TIME_UNIT_IN_SECONDS)
        a, b = validated_overlap(ti, tf)
        gyro = mapped_series(imu, GYRO_COLUMN, "gyroscope")
        force = (mapped_series(fp, VERTICAL_FORCE_COLUMN, "vertical force")
                 * FORCE_UPWARD_SIGN)
        valid_imu = (ti >= a) & (ti <= b)
        valid_force = (tf >= a) & (tf <= b)
        events = force_contact_events(
            tf[valid_force], force[valid_force],
            threshold_n=REFERENCE_THRESHOLD_N
        )
        ic = events.loc[events.event_type == "IC", "t_reference_s"].to_numpy(float)
        if len(ic) < 3:
            raise ValueError("Too few complete independent force contacts.")
        save_prepared_trial(cache_dir, key, ti[valid_imu],
                            gyro[valid_imu], ic)
        recorded.append(dict(
            subject_id=row.subject_id, session_id=row.session_id,
            condition=row.condition, trial_id=row.trial_id, trial_key=key,
            side=FOOT_SIDE, gyro_channel=GYRO_COLUMN,
            vertical_force_channel=VERTICAL_FORCE_COLUMN,
            n_reference_ic=len(ic), duration_s=b-a
        ))
        for ev in events.itertuples(index=False):
            event_rows.append(dict(trial_key=key,
                subject_id=row.subject_id, event_type=ev.event_type,
                t_reference_s=ev.t_reference_s, side=FOOT_SIDE))
    except (ValueError, FileNotFoundError, KeyError) as error:
        exclusions.append(dict(subject_id=row.subject_id,
            session_id=row.session_id, condition=row.condition,
            trial_id=row.trial_id, exclusion_reason=str(error)))
eligible = pd.DataFrame(recorded, columns=[
    "subject_id","session_id","condition","trial_id","trial_key",
    "side","gyro_channel","vertical_force_channel","n_reference_ic","duration_s"])
excluded = pd.DataFrame(exclusions, columns=[
    "subject_id","session_id","condition","trial_id","exclusion_reason"])
references = pd.DataFrame(event_rows, columns=[
    "trial_key","subject_id","event_type","t_reference_s","side"])

# ------------------------- 2. Save Scientific Tables -------------------------
eligible.to_csv(output_dir / "02_eligible_trials.csv", index=False)
excluded.to_csv(output_dir / "02_exclusions.csv", index=False)
references.to_csv(output_dir / "02_reference_events.csv", index=False)
display(eligible.head(), excluded.head())
print("Accepted trials:", len(eligible),
      "Independent force IC events:", int((references.event_type == "IC").sum()))
if eligible.empty:
    raise ValueError(
        "No aligned eligible IMU/force trials. Review 02_exclusions.csv; "
        "do not calculate detector timing error without independent references."
    )


**04. Create Participant-Disjoint Development and Test Roles**

Reserve entire people before model tuning. The split size and available fold count depend on **observed eligible participants**, not an assumed fixed 22-person cohort. Pilot-scale runs with fewer than six people cannot be presented as final grouped evaluation.


In [ ]:
# ======================= 04. Assign Participant Roles ========================

# --------------------- 1. Stage Inputs and Main Analysis ---------------------
people = eligible.subject_id.unique()
if len(people) < 7:
    print("Source pilot complete for", len(people),
          "people. Download additional valid participants before grouped "
          "Development/Test assignment and Notebooks 04–05.")
    roles = pd.DataFrame(columns=["subject_id","role"])
    folds = pd.DataFrame(columns=["subject_id","validation_fold"])
else:
    roles = participant_roles(people, test_fraction=0.2, seed=42)
    folds = development_folds(roles, n_splits=5)
    print("Role counts:", roles.role.value_counts().to_dict())
    display(roles, folds)

# ------------------------- 2. Save Scientific Tables -------------------------
roles.to_csv(output_dir / "02_participant_roles.csv", index=False)
folds.to_csv(output_dir / "02_development_folds.csv", index=False)


**05. Inspect Source-Aligned Example and Document Limits**

Plot one eligible original trial with IMU and force-derived contact times, but do not convert an unverified time shift into an optimised offset. Reference IC events are an algorithmically derived measurement, not raw force observations.


In [ ]:
# ========================= 05. Inspect Aligned Trial =========================

# --------------------- 1. Stage Inputs and Main Analysis ---------------------
from gait_experiments import read_prepared_trial
sample = eligible.iloc[0]
t, g, ic = read_prepared_trial(cache_dir, sample.trial_key)
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(t, g, label=f"IMU: {GYRO_COLUMN}", lw=1)
for j, event in enumerate(ic):
    ax.axvline(event, color="darkorange", alpha=0.3,
               label="Force-derived IC" if j == 0 else None)
ax.set(xlabel="Reviewed original time (s)",
       ylabel="Gyroscope (source unit)", title="Eligible paired trial")
ax.legend()

# ------------------------ 2. Export Scientific Figure ------------------------
fig.tight_layout()
save_figure(fig, output_dir / "02_imu_reference_alignment.png")
plt.show()
print("Prepared numerical caches:", cache_dir)
print("Output reference events:", output_dir / "02_reference_events.csv")


**06. Research Summary and Output Download**

Source-derived force contacts and eligible paired trials require a reviewed time/side/channel contract. This ZIP bundles only the current `02_` tables and diagnostic chart; original sensor data and numerical trial caches remain outside it. A one-person pilot does not establish a final grouped cohort.

The shared helper packages the saved `02_` research tables and figures into one ZIP **outside the repository**. Retrieve it from Colab's file browser. Raw Camargo archives are not included, and no output is pushed to GitHub automatically.


In [ ]:
# ================= 06. Research Summary and Output Download ==================

# ------------------- 1. Review Current Scientific Outputs --------------------
print('Reference and eligibility evidence saved to:', output_dir)

# ------------------- 2. Package Notebook-Specific Evidence -------------------
download_outputs_from_colab(
    folder="gait_analysis/output",
    stage=f"{NOTEBOOK_NUMBER:02d}",
)
